In [1]:
import os 
import numpy as np
import tensorflow as tf
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm
from transformers import BertTokenizerFast, AutoTokenizer
from datasets import load_dataset
import glob

2025-02-27 14:15:11.461535: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-27 14:15:11.479738: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740645911.499294 4176959 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740645911.504908 4176959 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-27 14:15:11.526735: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
tamil_file = "Tamil_Corpus_Cleaned.txt"

with open(tamil_file, encoding='utf-8') as f:  
    print(f.read(1000))

ஒலிம்பிக் போட்டிகள் நடந்த இடங்கள் 1. 1896 - ஏதென்ஸ், கிரீஸ் 2. 1900 - பாரிஸ், பிரான்ஸ் 3. 1904 - செயின் லூயிஸ், அமெரிக்கா 4. 1908 - லண்டன்,பிரிட்டன் 5. 1912 - ஸ்டோக்ஹோம், சுவீடன் 6. 1920 - ஆண்ட்வெர்ப், பெல்ஜியம் 7. 1924 - பாரிஸ், பிரான்ஸ் 8. 1928 - ஆம்ஸ்டர்டாம், ஹாலந்து 9. 1932 - லாஸ், ஏஞ்சல்ஸ் 10. 1936 - பெர்லின், ஜெர்மனி 11. 1948 - லண்டன், இங்கிலாந்து 12. 1952 - ஹல்சின்கி, பின்லாந்து 13. 1956 - மேபோர்ன்,ஆஸ்திரேலியா 14. 1960 - ரோம், இத்தாலி 15. 1964 - டோக்கியோ, ஜப்பான் 16. 1968 - மெக்சிகோ, மெக்ஸிக்கோ 17. 1972 - மியூனிக், ஜெர்மனி 18. 1976 - மான்ட்ரியல், கனடா 19. 1980 - மாஸ்கோ, USSR 20. 1984 - லாஸ் ஏஞ்சல்ஸ், அமெரிக்கா 21. 1988 - சியோல், தென் கொரியா 22. 1992 - பார்சிலோனா, ஸ்பெயின் 23. 1996 - அட்லாண்டா, அமெரிக்கா 24. 2000 - சிட்னி, ஆஸ்திரேலியா 25. 2004 - ஏதென்ஸ், கிரீஸ் 26. 2008 - பீஜிங், சீனா 27. 2012 - லண்டன், இங்கிலாந்து 28. 2016 - ரியோ, பிரேசில் 29. 2020 - டோக்கியோ, ஜப்பான் 30. 2024 - பாரிஸ், பிரான்ஸ் 31. 2028 - லாஸ் ஏஞ்சல்ஸ், அமெரிக்கா Music ncs:
இன்னும் வாழ்வதில் நம்பிக்கையற்றுப்போ

In [3]:
from datasets import load_dataset

dataset = load_dataset('text', data_files="Tamil_Corpus_Cleaned.txt", encoding="utf-8")

In [4]:
dataset['train']

Dataset({
    features: ['text'],
    num_rows: 6971580
})

In [5]:
#loading bert tokenizer to work as a base for the new tokenizer
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

def batch_iterator(batch_size=10000):
    for i in tqdm(range(0, len(dataset['train']), batch_size)):
        yield dataset['train'][i: i +batch_size]['text']
bert_tokenizer = tokenizer.train_new_from_iterator(text_iterator=batch_iterator(), 
                                                   vocab_size=20000
                                                   #, special_tokens=['[CLS]', '[PAD]','[SEP]','[UNK]','[MASK]']
                                                  )
bert_tokenizer.save_pretrained('bert_bbpe/')

100%|█████████████████████████████████████████████████████████████████████████████████| 698/698 [04:53<00:00,  2.38it/s]


('bert_bbpe/tokenizer_config.json',
 'bert_bbpe/special_tokens_map.json',
 'bert_bbpe/vocab.json',
 'bert_bbpe/merges.txt',
 'bert_bbpe/added_tokens.json',
 'bert_bbpe/tokenizer.json')

In [6]:
import tokenizers
from transformers import Trainer, TrainingArguments, LineByLineTextDataset, BertModel
from transformers import BertConfig, BertForMaskedLM, DataCollatorForLanguageModeling
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert_bbpe/')
max_seq_length = 64

def encode_with_truncation(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length",
                   max_length=max_seq_length, return_special_tokens_mask=True)


dataset = dataset["train"].map(encode_with_truncation, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

display(len(dataset))

Map:   0%|          | 0/6971580 [00:00<?, ? examples/s]

6971580

In [7]:
dataset.features

{'text': Value(dtype='string', id=None),
 'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None),
 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None),
 'special_tokens_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None)}

In [8]:
dataset = dataset.remove_columns(["text"])

In [9]:
dataset.save_to_disk("tokenized_dataset_bbpe/dataset")

Saving the dataset (0/6 shards):   0%|          | 0/6971580 [00:00<?, ? examples/s]

In [1]:
from transformers import Trainer, TrainingArguments, LineByLineTextDataset, BertModel
from transformers import BertConfig, BertForMaskedLM, DataCollatorForLanguageModeling
from transformers import AutoTokenizer
from datasets import load_from_disk 

from datasets import load_dataset
import glob
import tokenizers
from transformers import Trainer, TrainingArguments, LineByLineTextDataset, BertModel
from transformers import BertConfig, BertForMaskedLM, DataCollatorForLanguageModeling
from transformers import AutoTokenizer


dataset = load_from_disk("tokenized_dataset_bbpe/dataset")

2025-02-20 18:17:13.984418: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-20 18:17:14.041898: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740055634.085104 2401146 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740055634.097155 2401146 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-20 18:17:14.157761: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
tokenizer = AutoTokenizer.from_pretrained('bert_bbpe/')
# Model configuration optimized for efficiency
config = BertConfig(
    vocab_size=20000,  
    hidden_size=192,  
    num_hidden_layers=3,  
    num_attention_heads=3,  
    max_position_embeddings=64
)

# Initialize model
model = BertForMaskedLM(config)
print(f"Total Parameters: {model.num_parameters()}")

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Total Parameters: 7906208


In [3]:
import glob
import tokenizers

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True,
                                               mlm_probability=0.15)
epochs = 1
save_steps = 10000 #save checkpoint every 10000 steps
batch_size = 64

training_args = TrainingArguments(
    output_dir = 'bert_bbpe/',
    overwrite_output_dir=True,
    num_train_epochs = epochs,
    per_device_train_batch_size = batch_size,
    save_steps = save_steps,
    save_total_limit = 2, #only save the last 5 checkpoints
    fp16=True,
    learning_rate = 1e-4,  # 5e-5 is the default
    logging_steps = 5_000,
    # gradient_accumulation_steps=4,
    # gradient_checkpointing=True
)

trainer = Trainer(
    model = model,
    args = training_args,
    data_collator=data_collator,
    train_dataset=dataset

)

# Train model
trainer.train()

# Save final model
trainer.save_model('bert_bbpe/')

Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/home/hpc_users/2020s18055@stu.cmb.ac.lk/.local/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
5000,3.034800
10000,2.081300
15000,1.870600


/home/hpc_users/2020s18055@stu.cmb.ac.lk/.local/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


## Vocabualry 50000

In [5]:
#loading bert tokenizer to work as a base for the new tokenizer
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

def batch_iterator(batch_size=10000):
    for i in tqdm(range(0, len(dataset['train']), batch_size)):
        yield dataset['train'][i: i +batch_size]['text']
bert_tokenizer = tokenizer.train_new_from_iterator(text_iterator=batch_iterator(), 
                                                   vocab_size=50000
                                                   #, special_tokens=['[CLS]', '[PAD]','[SEP]','[UNK]','[MASK]']
                                                  )
bert_tokenizer.save_pretrained('bert_bbpe_50000/')

100%|█████████████████████████████████████████████████████████████████████████████████| 698/698 [03:40<00:00,  3.16it/s]


('bert_bbpe_50000/tokenizer_config.json',
 'bert_bbpe_50000/special_tokens_map.json',
 'bert_bbpe_50000/vocab.json',
 'bert_bbpe_50000/merges.txt',
 'bert_bbpe_50000/added_tokens.json',
 'bert_bbpe_50000/tokenizer.json')

In [6]:
import tokenizers
from transformers import Trainer, TrainingArguments, LineByLineTextDataset, BertModel
from transformers import BertConfig, BertForMaskedLM, DataCollatorForLanguageModeling
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert_bbpe_50000/')
max_seq_length = 64

def encode_with_truncation(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length",
                   max_length=max_seq_length, return_special_tokens_mask=True)


dataset = dataset["train"].map(encode_with_truncation, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

display(len(dataset))

Map:   0%|          | 0/6971580 [00:00<?, ? examples/s]

6971580

In [7]:
dataset.features

{'text': Value(dtype='string', id=None),
 'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None),
 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None),
 'special_tokens_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None)}

In [8]:
dataset = dataset.remove_columns(["text"])

In [9]:
dataset.save_to_disk("tokenized_dataset_bbpe_50000/dataset")

Saving the dataset (0/6 shards):   0%|          | 0/6971580 [00:00<?, ? examples/s]

In [3]:
from datasets import load_from_disk
from datasets import load_dataset
import glob
import tokenizers
from transformers import Trainer, TrainingArguments, LineByLineTextDataset, BertModel
from transformers import BertConfig, BertForMaskedLM, DataCollatorForLanguageModeling
from transformers import AutoTokenizer


dataset = load_from_disk("tokenized_dataset_bbpe_50000/dataset")

In [4]:
tokenizer = AutoTokenizer.from_pretrained('bert_bbpe_50000/')
# Model configuration optimized for efficiency
config = BertConfig(
    vocab_size=50000,  
    hidden_size=192,  
    num_hidden_layers=3,  
    num_attention_heads=3,  
    max_position_embeddings=64
)

# Initialize model
model = BertForMaskedLM(config)
print(f"Total Parameters: {model.num_parameters()}")

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Total Parameters: 13696208


In [5]:
import glob
import tokenizers

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True,
                                               mlm_probability=0.15)
epochs = 1
save_steps = 10000 #save checkpoint every 10000 steps
batch_size = 64

training_args = TrainingArguments(
    output_dir = 'bert_bbpe_50000/',
    overwrite_output_dir=True,
    num_train_epochs = epochs,
    per_device_train_batch_size = batch_size,
    save_steps = save_steps,
    save_total_limit = 2, #only save the last 5 checkpoints
    fp16=True,
    learning_rate = 1e-4,  # 5e-5 is the default
    logging_steps = 5_000,
    # gradient_accumulation_steps=4,
    # gradient_checkpointing=True
)

trainer = Trainer(
    model = model,
    args = training_args,
    data_collator=data_collator,
    train_dataset=dataset

)

# Train model
trainer.train()

# Save final model
trainer.save_model('bert_bbpe_50000/')

Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss
5000,2.986800
10000,1.918100
15000,1.738400
